In [ ]:
# Parte 1: Preparación del entorno
#!pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


In [ ]:
# Parte 2: Carga y análisis exploratorio de los datos
df = pd.read_csv('Mall_Customers.csv')

print(df.head())
print(df.info())
print(df.describe())

# Visualización de la distribución
sns.pairplot(df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']])
plt.show()


**¿Por qué es necesario escalar para K-Means y DBSCAN?**

Ambos algoritmos usan distancias (por ejemplo, distancia euclidiana) para agrupar los datos. Si una variable tiene un rango mucho mayor que otra, dominará la distancia total y, por lo tanto, tendrá más peso en la decisión del algoritmo, aunque no necesariamente sea más importante.

Ejemplo:
* Age: varía entre 18 y 70.

* Annual Income (k$): varía entre 15 y 137.

* Spending Score (1-100): varía entre 1 y 99.


Si no escalas, Annual Income influirá más en el cálculo de distancias simplemente porque sus valores son más grandes.

In [ ]:
# Parte 3: Preprocesamiento
X = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


**¿Qué es K-Means Clustering?**

Es un algoritmo no supervisado que divide los datos en K grupos (clusters) con base en la distancia entre puntos. Cada grupo se forma alrededor de un centroide. El objetivo es minimizar la distancia total entre los puntos y su centroide.

In [ ]:
# Parte 4.1: Clustering con K-Means
inertia = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

# Método del Codo
plt.plot(range(1, 10), inertia, marker='o')
plt.xlabel('Número de Clusters') # Numero de grupos
plt.ylabel('Inercia') # la suma de las distancias cuadradas de cada punto a su centroide (cuánto error hay dentro de cada cluster).
plt.title('Método del Codo para K-Means')
plt.show()

# Ajuste del modelo con K óptimo (ej. 4)
kmeans = KMeans(n_clusters=4, random_state=42)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

sns.scatterplot(x=X_scaled[:, 1], y=X_scaled[:, 2], hue=df['KMeans_Cluster'], palette='tab10')
plt.title('Segmentación por K-Means')
plt.show()


In [ ]:
# Parte 4.1.1: Interpretación de Clustering con K-Means
df.groupby('KMeans_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()


**¿Cómo sabemos si k=4 es realmente el mejor número de clusters?**

El método del codo nos da una pista visual, pero no una respuesta cuantitativa. El **Silhouette Score** nos da un número entre -1 y 1 que mide:

* Qué tan **cohesionado** está un punto con su propio cluster (distancia intra-cluster pequeña)
* Qué tan **separado** está del cluster más cercano (distancia inter-cluster grande)

Un valor cercano a **1** indica clusters bien definidos. Cercano a **0**, solapamiento. Negativo, posible error de asignación.

In [ ]:
# Parte 4.1.2: Evaluación de K-Means con Silhouette Score
from sklearn.metrics import silhouette_score

# Comparar silhouette para diferentes valores de k
silhouette_scores = []
k_values = range(2, 9)
for k in k_values:
    km_temp = KMeans(n_clusters=k, random_state=42)
    labels_temp = km_temp.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels_temp)
    silhouette_scores.append(score)
    print(f'k={k} → Silhouette Score: {score:.4f}')

plt.plot(k_values, silhouette_scores, marker='o', color='darkred')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. k para K-Means')
plt.axhline(y=max(silhouette_scores), color='gray', linestyle='--', alpha=0.5)
plt.show()

# Score del modelo final (k=4)
score_final = silhouette_score(X_scaled, df['KMeans_Cluster'])
print(f'\nSilhouette Score del modelo final (k=4): {score_final:.4f}')
print('Interpretación: valores > 0.5 indican estructura de cluster razonable.')

¿Qué es DBSCAN?

DBSCAN es un algoritmo de clustering basado en densidad. A diferencia de K-Means (que requiere que definas el número de clusters k), DBSCAN:

* Encuentra agrupaciones densas de puntos sin necesidad de saber cuántos clusters hay.

* Identifica automáticamente "ruido" o "outliers", etiquetándolos como -1.

**Reflexión: ¿Por qué no incluimos CustomerID ni Gender en el modelo?**

* **CustomerID**: Es un identificador único sin significado estadístico. Incluirlo distorsionaría el cálculo de distancias porque sus valores numéricos son arbitrarios.

* **Gender**: Es una variable categórica nominal. Para incluirla habría que codificarla (por ejemplo, con One-Hot Encoding), lo que añadiría una dimensión más al espacio de features. En este laboratorio la omitimos para simplificar y porque el dataset es pequeño (200 filas), pero en un análisis completo sería una variable a considerar.

> En Machine Learning, la selección de features siempre debe estar justificada, no implícita.

**¿Cómo elegimos eps y min_samples en DBSCAN?**

A diferencia de K-Means donde usamos el método del codo, en DBSCAN usamos la **gráfica k-distance** para encontrar el valor de `eps`:

1. Para cada punto, calculamos la distancia a su k-ésimo vecino más cercano (usamos k = min_samples)
2. Ordenamos esas distancias de mayor a menor
3. El **codo** de esa curva sugiere un buen valor de `eps`

> Regla práctica para `min_samples`: al menos `dimensiones + 1`. Con 3 features, mínimo 4 o 5.

In [ ]:
# Parte 4.2.0: Selección de parámetros para DBSCAN (k-distance graph)
from sklearn.neighbors import NearestNeighbors

min_samples = 5
nbrs = NearestNeighbors(n_neighbors=min_samples).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)

# Distancia al k-ésimo vecino, ordenada de mayor a menor
kth_distances = np.sort(distances[:, min_samples - 1])[::-1]

plt.plot(kth_distances, color='darkred')
plt.axhline(y=0.6, color='blue', linestyle='--', label='eps=0.6 (elegido)')
plt.xlabel('Puntos ordenados')
plt.ylabel('Distancia al 5° vecino más cercano')
plt.title('K-Distance Graph — Elección de eps para DBSCAN')
plt.legend()
plt.show()

print('El codo visible en la curva sugiere eps ≈ 0.6, por eso elegimos ese valor.')

In [ ]:
# Parte 4.2: Clustering con DBSCAN
dbscan = DBSCAN(eps=0.6, min_samples=5) # eps = radio máximo para considerar que dos puntos están “cerca”./ min_samples =  número mínimo de puntos vecinos necesarios para formar un cluster.
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

sns.scatterplot(x=X_scaled[:, 1], y=X_scaled[:, 2], hue=df['DBSCAN_Cluster'], palette='tab10')
plt.title('Segmentación por DBSCAN')
plt.show()


In [ ]:
# Parte 4.2.1: Interpretando los Clustering con DBSCAN
df['DBSCAN_Cluster'].value_counts()
df.groupby('DBSCAN_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()





In [ ]:
# Parte 4.2.1: Interpretando los Clustering con DBSCAN sin atípicos
df_no_noise = df[df['DBSCAN_Cluster'] != -1]
df_no_noise.groupby('DBSCAN_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()

**Evaluación de DBSCAN con Silhouette Score**

Para DBSCAN excluimos los puntos de ruido (etiqueta -1) del cálculo, ya que no pertenecen a ningún cluster y sesgarían la métrica.

In [ ]:
# Parte 4.2.2: Evaluación de DBSCAN con Silhouette Score
from sklearn.metrics import silhouette_score

# Solo evaluar sobre puntos que NO son ruido
mask_no_noise = df['DBSCAN_Cluster'] != -1
n_noise = (df['DBSCAN_Cluster'] == -1).sum()
n_clusters_dbscan = df['DBSCAN_Cluster'][mask_no_noise].nunique()

print(f'Puntos clasificados como ruido (-1): {n_noise}')
print(f'Clusters encontrados (sin ruido): {n_clusters_dbscan}')

if n_clusters_dbscan > 1:
    score_dbscan = silhouette_score(
        X_scaled[mask_no_noise],
        df['DBSCAN_Cluster'][mask_no_noise]
    )
    print(f'Silhouette Score DBSCAN (sin ruido): {score_dbscan:.4f}')
else:
    print('Solo 1 cluster → Silhouette Score no aplicable con estos parámetros.')

print('\n¿Por qué DBSCAN encuentra menos clusters que K-Means?')
print('DBSCAN agrupa según densidad, por lo que regiones poco densas quedan como ruido,')
print('en lugar de forzarse a pertenecer a un cluster como lo haría K-Means.')

**¿Qué es PCA?**

PCA (Principal Component Analysis) es una técnica matemática que:

* Reduce el número de variables (dimensiones) de un dataset.

* Conserva la mayor parte de la variabilidad (información) de los datos originales.

* Transforma los datos a componentes principales que son combinaciones lineales de las variables originales.

¿Qué significa reducir de 3 a 2 dimensiones?

Reduces de:

Tres variables originales (Edad, Ingreso, Gasto)

a:

Dos nuevas variables artificiales, llamadas Componentes principales: PC1 y PC2.

Cada una de estas componentes es una combinación lineal de las variables originales. Por ejemplo:



* PC1=0.5⋅Edad+0.7⋅Ingreso+0.5⋅Gasto


* PC2=−0.6⋅Edad+0.4⋅Ingreso+0.7⋅Gasto

**¿Cuánta información conservamos al reducir de 3 a 2 dimensiones con PCA?**

La `explained_variance_ratio_` nos dice qué proporción de la varianza total captura cada componente principal. Esto es fundamental: si dos componentes capturan el 90%+ de la varianza, la reducción es segura. Si capturan el 50%, estamos perdiendo demasiada información.

> **Componente principal ≠ variable original.** PC1 y PC2 son combinaciones lineales de todas las variables originales (Edad, Ingreso, Gasto), ponderadas por sus loadings.

In [ ]:
# Parte 5.1: Reducción de dimensionalidad con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['KMeans_Cluster'], cmap='tab10')
plt.title('Visualización PCA de Clusters K-Means')
plt.show()


In [ ]:
# Parte 5.1.0: Varianza explicada por PCA
print('Varianza explicada por componente:')
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {ratio:.4f} ({ratio*100:.1f}%)')

print(f'\nVarianza total conservada en 2D: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Gráfica de varianza explicada
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Varianza por componente
axes[0].bar(['PC1', 'PC2'], pca.explained_variance_ratio_ * 100, color=['#8B0000', '#C0392B'])
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_title('Varianza por componente principal')
for i, v in enumerate(pca.explained_variance_ratio_):
    axes[0].text(i, v*100 + 0.5, f'{v*100:.1f}%', ha='center', fontweight='bold')

# Varianza acumulada
axes[1].plot(['PC1', 'PC2'], np.cumsum(pca.explained_variance_ratio_) * 100,
             marker='o', color='darkred', linewidth=2)
axes[1].axhline(y=90, color='gray', linestyle='--', label='Umbral 90%')
axes[1].set_ylabel('Varianza explicada acumulada (%)')
axes[1].set_title('Varianza acumulada — ¿cuándo es suficiente?')
axes[1].legend()

plt.tight_layout()
plt.show()

# Loadings: qué variable aporta más a cada componente
print('\nLoadings (contribución de cada variable a cada componente):')
loadings = pd.DataFrame(
    pca.components_.T,
    index=['Age', 'Annual Income (k$)', 'Spending Score (1-100)'],
    columns=['PC1', 'PC2']
)
print(loadings.round(4))

In [ ]:
# Parte 5.1.1: Interpretación de Reducción de dimensionalidad con PCA
df.groupby('KMeans_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()


In [ ]:
# Parte 5.1.2: Interpretación detallada de Reducción de dimensionalidad con PCA
# Mostrar algunos clientes por cluster
for i in range(4):
    print(f"\nCluster {i}")
    display(df[df['KMeans_Cluster'] == i][['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].head())


**¿Qué es t-SNE?**

t-SNE es una técnica de reducción de dimensionalidad no lineal diseñada específicamente para visualización de datos complejos en 2D o 3D.

A diferencia de PCA (que usa combinaciones lineales de variables para preservar la varianza), t-SNE preserva la estructura local de los datos. Es decir:

Los puntos que estaban cerca en el espacio original seguirán cerca en el nuevo espacio 2D.


In [ ]:
# Parte 5.2: Reducción de dimensionalidad con t-SNE
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42) # cuántos vecinos cercanos considera t-SNE / cuánto se mueven los puntos
X_tsne = tsne.fit_transform(X_scaled)

plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=df['KMeans_Cluster'], cmap='tab10')
plt.title('Visualización t-SNE de Clusters K-Means')
plt.show()


In [ ]:
# Parte 5.2.1: Interpretación de Reducción de dimensionalidad con t-SNE
df.groupby('KMeans_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()


## Parte 7: Detección de Anomalías

**¿Qué es la detección de anomalías?**

La detección de anomalías identifica puntos de datos que se comportan de forma significativamente distinta al resto. En entornos de negocio esto es crítico para:

* Detección de fraudes en transacciones
* Identificación de fallas en equipos (mantenimiento predictivo)
* Ciberseguridad: patrones de acceso inusuales

**Métodos que veremos:**

1. **Isolation Forest** → Aísla anomalías construyendo árboles de decisión aleatorios. Los puntos anómalos requieren menos cortes para ser aislados.
2. **LOF (Local Outlier Factor)** → Mide la densidad local de cada punto vs. sus vecinos. Un punto con densidad muy baja respecto a sus vecinos es considerado anomalía.

> Tanto Isolation Forest como LOF son métodos **no supervisados**: no necesitan etiquetas previas.

In [ ]:
# Parte 7.1: Detección de Anomalías con Isolation Forest
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    contamination=0.05,  # Asumimos que ~5% de los datos podrían ser anomalías
    random_state=42
)
df['Anomaly_IF'] = iso_forest.fit_predict(X_scaled)
# fit_predict devuelve: 1 = normal, -1 = anomalía

n_anomalias = (df['Anomaly_IF'] == -1).sum()
print(f'Puntos normales: {(df["Anomaly_IF"] == 1).sum()}')
print(f'Anomalías detectadas: {n_anomalias} ({n_anomalias/len(df)*100:.1f}%)')

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ingreso vs Gasto
colors = df['Anomaly_IF'].map({1: '#2ecc71', -1: '#e74c3c'})
axes[0].scatter(X_scaled[:, 1], X_scaled[:, 2], c=colors, alpha=0.7, edgecolors='k', linewidth=0.3)
axes[0].set_xlabel('Annual Income (escalado)')
axes[0].set_ylabel('Spending Score (escalado)')
axes[0].set_title('Isolation Forest — Ingreso vs Gasto')
axes[0].legend(handles=[
    plt.scatter([], [], c='#2ecc71', label='Normal'),
    plt.scatter([], [], c='#e74c3c', label='Anomalía')
], loc='upper left')

# Edad vs Ingreso
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=colors, alpha=0.7, edgecolors='k', linewidth=0.3)
axes[1].set_xlabel('Age (escalado)')
axes[1].set_ylabel('Annual Income (escalado)')
axes[1].set_title('Isolation Forest — Edad vs Ingreso')

plt.tight_layout()
plt.show()

# ¿Quiénes son las anomalías?
print('\nPerfiles de los clientes detectados como anomalías:')
display(df[df['Anomaly_IF'] == -1][['Age', 'Annual Income (k$)', 'Spending Score (1-100)']])

In [ ]:
# Parte 7.2: Detección de Anomalías con LOF (Local Outlier Factor)
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,       # Cuántos vecinos considera para medir densidad local
    contamination=0.05    # Proporción esperada de anomalías
)
df['Anomaly_LOF'] = lof.fit_predict(X_scaled)
# fit_predict devuelve: 1 = normal, -1 = anomalía

n_lof = (df['Anomaly_LOF'] == -1).sum()
print(f'Anomalías detectadas por LOF: {n_lof}')

# Comparación IF vs LOF
ambos = ((df['Anomaly_IF'] == -1) & (df['Anomaly_LOF'] == -1)).sum()
print(f'Detectadas por AMBOS métodos (mayor confianza): {ambos}')

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
                           ['Anomaly_IF', 'Anomaly_LOF'],
                           ['Isolation Forest', 'LOF']):
    c = df[col].map({1: '#2ecc71', -1: '#e74c3c'})
    ax.scatter(X_scaled[:, 1], X_scaled[:, 2], c=c, alpha=0.7,
               edgecolors='k', linewidth=0.3)
    ax.set_xlabel('Annual Income (escalado)')
    ax.set_ylabel('Spending Score (escalado)')
    ax.set_title(f'{title} — Ingreso vs Gasto')

plt.tight_layout()
plt.show()

**¿Cómo interpretar las anomalías encontradas?**

Los clientes marcados como anomalías tienen combinaciones inusuales de edad, ingreso y gasto. Por ejemplo, un cliente con ingreso muy alto y gasto muy bajo podría ser un outlier de negocio relevante (¿cliente VIP que no compra?), o simplemente ruido en los datos.

**Diferencia entre los dos métodos:**

| Método | Enfoque | Ventaja | Limitación |
|---|---|---|---|
| Isolation Forest | Aislamiento por árboles | Escala bien con datos grandes | Menos preciso en zonas densas |
| LOF | Densidad local vs vecinos | Sensible a anomalías locales | Costoso computacionalmente en datasets grandes |

> En la práctica, usar **ambos métodos y comparar** es buena práctica: los puntos detectados por los dos tienen mayor probabilidad de ser anomalías reales.

## Parte 8: Análisis Comparativo y Conclusiones

En esta sección sintetizamos los hallazgos de todos los modelos aplicados al dataset Mall Customers.

In [ ]:
# Parte 8: Resumen comparativo de todos los modelos
print('=' * 60)
print('RESUMEN COMPARATIVO — MODELOS NO SUPERVISADOS')
print('Dataset: Mall Customers (200 clientes)')
print('Features usadas: Age, Annual Income (k$), Spending Score (1-100)')
print('=' * 60)

from sklearn.metrics import silhouette_score

# K-Means
sk = silhouette_score(X_scaled, df['KMeans_Cluster'])
print(f'\n[K-Means] k=4 | Silhouette Score: {sk:.4f}')
print('  → Clusters bien definidos, separados por nivel de ingreso y gasto')

# DBSCAN
mask = df['DBSCAN_Cluster'] != -1
n_ruido = (~mask).sum()
n_cl = df['DBSCAN_Cluster'][mask].nunique()
print(f'\n[DBSCAN] eps=0.6, min_samples=5 | Clusters: {n_cl} | Ruido: {n_ruido} puntos')
if n_cl > 1:
    sd = silhouette_score(X_scaled[mask], df['DBSCAN_Cluster'][mask])
    print(f'  Silhouette Score (sin ruido): {sd:.4f}')
print('  → Identifica estructura densa; puntos periféricos quedan como outliers')

# PCA
var_acum = pca.explained_variance_ratio_.sum()
print(f'\n[PCA] 2 componentes | Varianza conservada: {var_acum*100:.1f}%')
print('  → Permite visualizar los clusters de K-Means en 2D con pérdida mínima de info')

# Anomalías
n_if = (df['Anomaly_IF'] == -1).sum()
n_lof = (df['Anomaly_LOF'] == -1).sum()
n_ambos = ((df['Anomaly_IF'] == -1) & (df['Anomaly_LOF'] == -1)).sum()
print(f'\n[Isolation Forest] Anomalías: {n_if} ({n_if/len(df)*100:.1f}%)')
print(f'[LOF]              Anomalías: {n_lof} ({n_lof/len(df)*100:.1f}%)')
print(f'[Consenso]         Detectadas por ambos: {n_ambos} — mayor confianza')

print('\n' + '=' * 60)
print('CONCLUSIONES')
print('=' * 60)
print('''
1. K-Means con k=4 produce la segmentación más interpretable,
   confirmada por el Silhouette Score y validada visualmente con PCA.

2. DBSCAN complementa K-Means al identificar clientes atípicos (ruido)
   que K-Means asigna forzosamente a un cluster. Esto es valioso para
   análisis de calidad de datos.

3. PCA con 2 componentes conserva suficiente varianza para visualización
   confiable. t-SNE revela la misma estructura preservando relaciones locales.

4. La detección de anomalías identifica un pequeño grupo de clientes con
   comportamiento atípico que merece investigación adicional de negocio.
''')